# 🚀 Day 4: Distributed AI Training, Slurm Multi-Node Clusters & Intel VTune Profiling Optimization
### GLA University — Department of Computer Science & Engineering | BCSG701
**Academic Partnership**: Intel Software & Supercomputing Cluster Lab

Welcome to **Day 4** of the 5-Day HPC Masterclass. Today we scale deep learning from single workstations to multi-node supercomputer clusters, orchestrate workloads with Slurm, analyze collective communication bottlenecks (Ring-AllReduce), profile microarchitectures with **Intel VTune (TMAM)**, and evaluate memory vs compute saturation using the **Intel Advisor Roofline Model**.

## 📌 Section 0: Environment Initialization & oneAPI Toolchain Check
Let's audit the compute environment, CPU topology, available memory channels, and install required distributed profiling libraries.

In [ ]:
!pip install -q matplotlib numpy torch psutil tabulate
import os, sys, psutil, torch, numpy as np, matplotlib.pyplot as plt
print(f"[INFO] Python Version   : {sys.version.split()[0]}")
print(f"[INFO] Logical Cores    : {os.cpu_count()}")
print(f"[INFO] Physical Cores   : {psutil.cpu_count(logical=False)}")
print(f"[INFO] Total System RAM : {psutil.virtual_memory().total / (1024**3):.2f} GB")
print(f"[INFO] PyTorch Version  : {torch.__version__}")
print(f"[INFO] oneDNN Enabled   : {torch.backends.mkldnn.is_available()}")

## 📋 Section 1: Multi-Node Slurm Resource Scheduling & Job Script Generator
In high-performance clusters, researchers never run commands directly on compute nodes. All batch jobs are submitted through Slurm (`sbatch`).
Below, we programmatically generate an enterprise-grade multi-node `sbatch` job script configured for 4 compute nodes with Intel oneCCL and process pinning.

In [ ]:
sbatch_script = '''#!/bin/bash
#SBATCH --job-name=gla_hpc_dist_4node
#SBATCH --partition=compute_cluster
#SBATCH --nodes=4
#SBATCH --ntasks-per-node=2
#SBATCH --cpus-per-task=16
#SBATCH --time=02:00:00
#SBATCH --output=slurm_job_%j.log
#SBATCH --error=slurm_job_%j.err

# 1. Load Intel oneAPI Environment Modules
source /opt/intel/oneapi/setvars.sh --force > /dev/null 2>&1

# 2. Thread Binding and NUMA Placement
export OMP_NUM_THREADS=$SLURM_CPUS_PER_TASK
export OMP_PLACES=cores
export OMP_PROC_BIND=close
export I_MPI_PIN_DOMAIN=auto
export I_MPI_FABRICS=shm:ofi
export HOROVOD_FUSION_THRESHOLD=67108864

# 3. Execute Distributed Multi-Node Training via MPI Process Manager
echo "[SLURM] Starting Job ID: $SLURM_JOB_ID on $(date)"
mpirun -np $SLURM_NTASKS -ppn $SLURM_NTASKS_PER_NODE python train_distributed.py
echo "[SLURM] Job Completed on $(date)"
'''
with open("submit_multinode_job.sh", "w") as f:
    f.write(sbatch_script)
print("[OK] Enterprise Slurm job script generated: submit_multinode_job.sh")
with open("submit_multinode_job.sh") as f:
    print(f.read())

## 🔄 Section 2: Baidu / Horovod Ring-AllReduce Engine Simulation
Parameter servers suffer an $O(P)$ network bandwidth bottleneck at scale. Ring-AllReduce eliminates this bottleneck by organizing workers in a logical ring, dividing gradient tensors into $P$ chunks, and executing in two phases:
1. **Scatter-Reduce Phase** ($P-1$ steps): Each worker accumulates one slice of the global gradient.
2. **AllGather Phase** ($P-1$ steps): The reduced slices circulate until all workers possess the complete averaged gradient.

Let's simulate and visualize this exact mathematical process across 4 simulated HPC nodes.

In [ ]:
def simulate_ring_allreduce(num_workers=4, tensor_size=8):
    P = num_workers
    chunk_size = tensor_size // P
    # Initialize distinct local gradient vectors on each worker
    # Worker 0: [1, 1, ...], Worker 1: [2, 2, ...], etc.
    buffers = [np.full(tensor_size, float(i + 1)) for i in range(P)]
    
    print(f"--- INITIAL WORKER GRADIENT BUFFERS (P={P} workers) ---")
    for i in range(P):
        print(f"Worker {i}: {buffers[i]}")
        
    # Phase 1: Scatter-Reduce (P - 1 steps)
    print("\n--- PHASE 1: SCATTER-REDUCE ---")
    for step in range(P - 1):
        new_buffers = [b.copy() for b in buffers]
        for i in range(P):
            chunk_idx = (i - step) % P
            recv_worker = (i + 1) % P
            start = chunk_idx * chunk_size
            end = start + chunk_size
            new_buffers[recv_worker][start:end] += buffers[i][start:end]
        buffers = new_buffers
        print(f"  Scatter-Reduce Step {step+1}/{P-1} complete.")
        
    # Phase 2: AllGather (P - 1 steps)
    print("\n--- PHASE 2: ALLGATHER ---")
    for step in range(P - 1):
        new_buffers = [b.copy() for b in buffers]
        for i in range(P):
            chunk_idx = (i + 1 - step) % P
            recv_worker = (i + 1) % P
            start = chunk_idx * chunk_size
            end = start + chunk_size
            new_buffers[recv_worker][start:end] = buffers[i][start:end]
        buffers = new_buffers
        print(f"  AllGather Step {step+1}/{P-1} complete.")
        
    # Average across workers (divide by P)
    for i in range(P):
        buffers[i] /= P
        
    print("\n--- FINAL SYNCHRONIZED BUFFERS (Global Average Gradients) ---")
    for i in range(P):
        print(f"Worker {i}: {buffers[i]}")
        
    # Verification
    expected = sum(float(i + 1) for i in range(P)) / P
    assert np.allclose(buffers[0], expected), "Verification Failed!"
    print(f"\n[VERIFIED] All {P} workers converged to exact global average: {expected}")

simulate_ring_allreduce(num_workers=4, tensor_size=8)


## 🔬 Section 3: Intel VTune Profiler & TMAM Metrics Analyzer
Intel's **Top-down Microarchitecture Analysis Method (TMAM)** categorizes CPU pipeline slots into four fundamental buckets:
1. **Retiring**: Productive work where instructions successfully executed and retired.
2. **Bad Speculation**: Cycles wasted due to branch predictor mispredictions.
3. **Frontend Bound**: Pipeline starved because instructions could not be fetched/decoded fast enough.
4. **Backend Bound**: Pipeline stalled waiting on Memory (DRAM/Caches) or Core execution units.

Let's write a parser that ingests VTune hardware counter telemetry and generates a professional diagnostic report.

In [ ]:
def analyze_vtune_tmam(retiring_pct, bad_spec_pct, frontend_pct, backend_memory_pct, backend_core_pct):
    total_backend = backend_memory_pct + backend_core_pct
    total = retiring_pct + bad_spec_pct + frontend_pct + total_backend
    
    print("===========================================================")
    print("      INTEL VTUNE PROFILER - TMAM LEVEL 1 & 2 REPORT       ")
    print("===========================================================")
    print(f"1. Retiring (Useful Work)        : {retiring_pct:5.1f}%  [Target: > 40.0%]")
    print(f"2. Bad Speculation (Flushed)     : {bad_spec_pct:5.1f}%  [Target: < 10.0%]")
    print(f"3. Frontend Bound (Fetch Stalls) : {frontend_pct:5.1f}%  [Target: < 15.0%]")
    print(f"4. Backend Bound (Execution)     : {total_backend:5.1f}%")
    print(f"   ├── Backend Memory Bound      : {backend_memory_pct:5.1f}%  (Cache / DRAM Stalls)")
    print(f"   └── Backend Core Bound        : {backend_core_pct:5.1f}%  (Divider / Port Congestion)")
    print("-----------------------------------------------------------")
    
    # Actionable diagnostic recommendations
    if backend_memory_pct > 30.0:
        print("[ALERT] SEVERE MEMORY BOTTLENECK DETECTED!")
        print("        Recommendation: Apply cache blocking/tiling, operator fusion,")
        print("        and PyTorch channels_last memory format to minimize DRAM stalls.")
    elif bad_spec_pct > 15.0:
        print("[ALERT] HIGH BRANCH MISPREDICTION RATE!")
        print("        Recommendation: Eliminate dynamic branches in hot loops, use SIMD masks.")
    elif retiring_pct >= 40.0:
        print("[SUCCESS] Code is executing efficiently with high retiring pipeline ratio.")
    print("===========================================================")

# Run diagnostic on unoptimized deep learning training loop
analyze_vtune_tmam(retiring_pct=21.4, bad_spec_pct=6.2, frontend_pct=8.1, backend_memory_pct=48.3, backend_core_pct=16.0)

## 📈 Section 4: Intel Advisor & Interactive Roofline Model
The **Roofline Model** maps application performance ($y$-axis in GFLOP/s) against its **Arithmetic Intensity** ($x$-axis in FLOP/Byte).
The model highlights whether code is bound by **Memory Bandwidth** (left of the knee) or by **Compute Peak** (right of the knee).
Below, we compute and render a live interactive Roofline plot comparing an unoptimized baseline against an optimized HPC kernel.

In [ ]:
def plot_roofline():
    # Target System Hardware Specs: Intel Xeon Platinum 8480+ (Dual Socket)
    peak_memory_bandwidth = 600.0  # GB/s (DDR5-4800 across 16 channels)
    peak_compute_fp32 = 12000.0   # GFLOP/s (AVX-512 vector units)
    peak_compute_amx  = 96000.0   # GFLOP/s (Intel AMX BF16 matrix tiles)
    
    # Arithmetic Intensity range from 0.01 to 1000 FLOP/Byte
    ai = np.logspace(-2, 3, 500)
    
    # Roofline ceilings
    roof_fp32 = np.minimum(peak_compute_fp32, ai * peak_memory_bandwidth)
    roof_amx  = np.minimum(peak_compute_amx,  ai * peak_memory_bandwidth)
    
    # Workload Points
    # Baseline unoptimized training loop: AI = 0.5 FLOP/Byte, Perf = 280 GFLOP/s
    # Optimized HPC kernel (AVX-512 + Fusion): AI = 16.0 FLOP/Byte, Perf = 8400 GFLOP/s
    # AMX BF16 Tensor Kernel: AI = 64.0 FLOP/Byte, Perf = 54000 GFLOP/s
    points = [
        ("Baseline Kernel (Memory Bound)", 0.5, 280.0, "red", "o"),
        ("AVX-512 Fused Kernel (Balanced)", 16.0, 8400.0, "blue", "s"),
        ("Intel AMX BF16 Kernel (Compute Bound)", 64.0, 54000.0, "green", "^")
    ]
    
    plt.figure(figsize=(10, 6))
    plt.loglog(ai, roof_fp32, 'b-', linewidth=2, label=f"Peak AVX-512 FP32 ({peak_compute_fp32/1000:.1f} TFLOP/s)")
    plt.loglog(ai, roof_amx, 'g--', linewidth=2, label=f"Peak Intel AMX BF16 ({peak_compute_amx/1000:.1f} TFLOP/s)")
    
    for label, pt_ai, pt_perf, color, marker in points:
        plt.loglog(pt_ai, pt_perf, marker=marker, color=color, markersize=10, label=f"{label} ({pt_perf:.0f} GFLOP/s)")
        plt.annotate(label, (pt_ai * 1.2, pt_perf), fontsize=9, fontweight='bold', color=color)
        
    # Annotate Memory Bandwidth Slope
    plt.text(0.05, 50, f"DRAM Bandwidth: {peak_memory_bandwidth} GB/s", rotation=38, fontsize=10, color='gray')
    
    plt.grid(True, which="both", ls="--", alpha=0.5)
    plt.title("Intel Advisor Roofline Model Analysis | GLA University HPC", fontsize=13, fontweight='bold')
    plt.xlabel("Arithmetic Intensity (FLOP/Byte)", fontsize=11)
    plt.ylabel("Attainable Performance (GFLOP/s)", fontsize=11)
    plt.legend(loc="lower right", fontsize=9)
    plt.tight_layout()
    plt.savefig("roofline_model_analysis.png", dpi=150)
    plt.show()
    print("[OK] Roofline plot rendered and saved to roofline_model_analysis.png")

plot_roofline()

## 📊 Section 5: Cluster Scaling Analysis: Strong vs Weak Scaling Simulator
Let's measure and plot cluster scaling efficiency across 1, 2, 4, 8, and 16 nodes comparing **Amdahl's Law (Strong Scaling)** and **Gustafson's Law (Weak Scaling)**.

In [ ]:
nodes = np.array([1, 2, 4, 8, 16])
serial_fraction = 0.05  # 5% serial synchronization overhead

# Strong Scaling (Amdahl's Law)
speedup_amdahl = 1.0 / (serial_fraction + (1.0 - serial_fraction) / nodes)
efficiency_amdahl = (speedup_amdahl / nodes) * 100.0

# Weak Scaling (Gustafson's Law)
speedup_gustafson = serial_fraction + nodes * (1.0 - serial_fraction)
efficiency_gustafson = (speedup_gustafson / nodes) * 100.0

# Simulated Measured Production Run with InfiniBand RDMA
measured_speedup = np.array([1.0, 1.94, 3.76, 7.12, 13.8])
measured_efficiency = (measured_speedup / nodes) * 100.0

plt.figure(figsize=(10, 5))
plt.plot(nodes, nodes, 'k--', label="Ideal Linear Scaling (100% Efficiency)")
plt.plot(nodes, speedup_amdahl, 'r-o', linewidth=2, label="Amdahl's Law (Strong Scaling, s=5%)")
plt.plot(nodes, speedup_gustafson, 'g-s', linewidth=2, label="Gustafson's Law (Weak Scaling, s=5%)")
plt.plot(nodes, measured_speedup, 'b-^', linewidth=2, label="Empirical HPC Cluster (Horovod + InfiniBand)")

plt.title("Multi-Node Distributed AI Scaling Comparison", fontsize=13, fontweight='bold')
plt.xlabel("Cluster Size (Number of Compute Nodes)", fontsize=11)
plt.ylabel("Effective Speedup Factor (X)", fontsize=11)
plt.xticks(nodes)
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(fontsize=10)
plt.tight_layout()
plt.savefig("cluster_scaling_comparison.png", dpi=150)
plt.show()

print("Cluster Scaling Metrics Table:")
print(f"{'Nodes':<8}{'Ideal':<10}{'Amdahl (X)':<14}{'Gustafson (X)':<16}{'Measured (X)':<14}{'Efficiency (%)':<15}")
for i, n in enumerate(nodes):
    print(f"{n:<8}{n:<10.1f}{speedup_amdahl[i]:<14.2f}{speedup_gustafson[i]:<16.2f}{measured_speedup[i]:<14.2f}{measured_efficiency[i]:<15.1f}")

## 🏆 Day 4 Mastery Verification & Summary
You have mastered:
1. Multi-node Slurm job scheduling (`sbatch`, PMIx, NUMA binding).
2. Mathematical mechanics of Ring-AllReduce (Scatter-Reduce + AllGather).
3. Intel VTune Profiler TMAM pipeline slot analysis (Retiring, Bad Speculation, Frontend, Backend Memory vs Core Bound).
4. Intel Advisor Roofline model construction and Arithmetic Intensity optimization.
5. Cluster scaling laws (Amdahl vs Gustafson) and InfiniBand RDMA acceleration.

👉 Next: **Day 5 — Generative AI, LLMs & Habana Gaudi Acceleration!**